# 1st LAB: Word Embeddings

In [1]:
import re
import string

import nltk
import numpy as np
import pandas as pd

# Loading the data

In [2]:
# US presidential inaugural speeches, from the NLTK corpora
nltk.download('inaugural') ###
from nltk.corpus import inaugural

[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!


In [3]:
# Extract the year, president's name and speech text from each file
data = pd.DataFrame({
    # Extract the year from the first 4 characters of each filename
    'year': [int(f[:4]) for f in inaugural.fileids()],
    # Extract the president's name from 6th char to last before ".txt"
    'president': [f[5:-4] for f in inaugural.fileids()],
    # Retrieve the full text of each inaugural speech
    'text': [inaugural.raw(f) for f in inaugural.fileids()],
}).sort_values('year', ignore_index=True)  # Sort speeches by year

data.head()

,year,president,text
0,1789,Washington,Fellow-Citizens of the Senate and of the House...
1,1793,Washington,"Fellow citizens, I am again called upon by the..."
2,1797,Adams,"When it was first perceived, in early times, t..."
3,1801,Jefferson,Friends and Fellow Citizens:\n\nCalled upon to...
4,1805,Jefferson,"Proceeding, fellow citizens, to that qualifica..."


In [4]:
# Define functions to remove punctuation and numbers from text
def remove_punctuation(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

def remove_numbers(text):
    return re.sub(r"\d+", "", text)

In [5]:
# Convert text to lowercase
data["preprocessed"] = data["text"].str.lower()

# Remove numbers and punctuation
data["preprocessed"] = data["preprocessed"].apply(remove_numbers)
data["preprocessed"] = data["preprocessed"].apply(remove_punctuation)

# Split text into individual words (tokens)
data["tokens"] = data["preprocessed"].str.split()

data.head()

,year,president,text,preprocessed,tokens
0,1789,Washington,Fellow-Citizens of the Senate and of the House...,fellow citizens of the senate and of the house...,"[fellow, citizens, of, the, senate, and, of, t..."
1,1793,Washington,"Fellow citizens, I am again called upon by the...",fellow citizens i am again called upon by the...,"[fellow, citizens, i, am, again, called, upon,..."
2,1797,Adams,"When it was first perceived, in early times, t...",when it was first perceived in early times t...,"[when, it, was, first, perceived, in, early, t..."
3,1801,Jefferson,Friends and Fellow Citizens:\n\nCalled upon to...,friends and fellow citizens \n\ncalled upon to...,"[friends, and, fellow, citizens, called, upon,..."
4,1805,Jefferson,"Proceeding, fellow citizens, to that qualifica...",proceeding fellow citizens to that qualifica...,"[proceeding, fellow, citizens, to, that, quali..."


In [6]:
print(len(data))

60


# Introduction to gensim

**📎 From the slide to the code** · slide *Word2Vec in Python / Words as vectors*

| In the slide | In the code |
|---|---|
| Train Word2Vec on a list of sentences | `Word2Vec(sentences=toy, ...)` |
| vector_size: how many numbers per word | `vector_size=10` |
| window: how many words around are the context | `window=2` |
| CBOW or skip-gram | `sg=0` / `sg=1` |
| Analogies: biggest − big + small | `most_similar(positive=[...], negative=[...])` |
| Frequent pairs become one token | `Phrases(...)` |

Before the exercises, let's see **how to use** the gensim tools on a toy corpus of 5 sentences.

> **BEWARE:** the corpus is far too small, so the results (vectors, similar words) mean nothing. Look only at **which functions** we call and **in which form** we pass the data.

In [7]:
from gensim.models import Word2Vec

# Word2Vec wants a LIST OF SENTENCES, each sentence a LIST OF WORDS
toy = [
    ['i', 'like', 'green', 'eggs', 'and', 'ham'],
    ['green', 'eggs', 'for', 'breakfast'],
    ['i', 'like', 'bacon', 'and', 'toast'],
    ['we', 'eat', 'green', 'eggs', 'with', 'toast'],
    ['the', 'king', 'eats', 'ham', 'and', 'bacon'],
]

# sg=1: skip-gram, sg=0: CBOW
# min_count=1: keep also the words seen only once (the default, 5, would remove all of them here)
toy_model = Word2Vec(sentences=toy, vector_size=10, window=2, min_count=1, sg=1, seed=42)

print('Vocabulary:', list(toy_model.wv.key_to_index))
print('Vector of eggs:', toy_model.wv['eggs'].round(2))
print('Most similar to eggs:', toy_model.wv.most_similar('eggs', topn=3))

# word analogies: positive words are added, negative words are subtracted (eggs + bacon - ham)
print('eggs + bacon - ham:', toy_model.wv.most_similar(positive=['eggs', 'bacon'], negative=['ham'], topn=3))

# a word that is not in the corpus has no vector
print("Is 'pizza' in the vocabulary?", 'pizza' in toy_model.wv)

Vocabulary: ['and', 'eggs', 'green', 'toast', 'bacon', 'ham', 'like', 'i', 'eats', 'king', 'the', 'with', 'eat', 'we', 'breakfast', 'for']
Vector of eggs: [ 0.01  0.1   0.05  0.05  0.04  0.06  0.   -0.07  0.07 -0.01]
Most similar to eggs: [('king', 0.3018323481082916), ('with', 0.2954557240009308), ('eats', 0.2678139805793762)]
eggs + bacon - ham: [('and', 0.6240828037261963), ('eat', 0.36743733286857605), ('with', 0.24361400306224823)]
Is 'pizza' in the vocabulary? False


### Bigrams with `Phrases`

`Phrases` finds the pairs of words that often appear together and joins them with `_`.

In [8]:
from gensim.models import Phrases

# min_count and threshold are very low only because the toy corpus is tiny: on real data keep the defaults
toy_phrases = Phrases(toy, min_count=1, threshold=1)

# apply it to the sentences: the result is a new corpus, in the same form (list of lists of words)
toy_bigrams = list(toy_phrases[toy])
print(toy_bigrams)

[['i_like', 'green_eggs', 'and', 'ham'], ['green_eggs', 'for', 'breakfast'], ['i_like', 'bacon', 'and', 'toast'], ['we', 'eat', 'green_eggs', 'with', 'toast'], ['the', 'king', 'eats', 'ham', 'and', 'bacon']]


### Document vectors with `Doc2Vec`

**📎 From the slide to the code** · slide *Doc2Vec in Python*

| In the slide | In the code |
|---|---|
| A document = its words + a number | `TaggedDocument(words, [i])` |
| The vector learned for document 0 | `toy_doc_model.dv[0]` |
| The vector of a new document | `toy_doc_model.infer_vector([...])` |

Many tasks work on **whole documents** (sentiment analysis, passage retrieval, clustering), so we need one vector per document.

- **First idea:** average the word vectors of the document, or better, weight them with TF-IDF (this is Exercise 3.3).
- **Doc2Vec** (Le & Mikolov, 2014): each document gets **its own vector**, learned together with the word vectors. It usually works better than averaging.

**How Doc2Vec learns.** Each document is treated as **one extra word**, a tag like `doc 1`, and Word2Vec is trained as usual:

Document 1: `"fry the eggs and the bacon"`

| | tag | context words | word to guess |
|---|---|---|---|
| window 1 | `doc 1` | `fry` `the` | `eggs` |
| window 2 | `doc 1` | `eggs` `and` | `the` |
| window 3 | `doc 1` | `and` `the` | `bacon` |

- The tag is added to **every window** of its document: `doc 1` sees all the words of document 1, so its vector absorbs the **topic** of the whole document (here: near `eggs`, `bacon`, `fry`).
- A **new document** (not seen in training): the word vectors stay fixed and only its tag is learned. This is what `infer_vector` does.

In gensim each document is a `TaggedDocument`: the list of its words + a tag (here, its number). After training, `model.dv[i]` is the vector of document `i`.

In [9]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

toy_docs = [TaggedDocument(words, [i]) for i, words in enumerate(toy)]
print(toy_docs[0])

toy_doc_model = Doc2Vec(toy_docs, vector_size=10, window=2, min_count=1, seed=42)

# the vector learned for a document of the corpus, by its tag
print('Vector of document 0:', toy_doc_model.dv[0].round(2))

# the vector of a NEW text (a list of words, not a string)
print('Vector of a new text:', toy_doc_model.infer_vector(['green', 'eggs', 'and', 'bacon']).round(2))

TaggedDocument<['i', 'like', 'green', 'eggs', 'and', 'ham'], [0]>
Vector of document 0: [ 0.08  0.09  0.08 -0.08 -0.08 -0.05 -0.04  0.05 -0.07 -0.08]
Vector of a new text: [-0.04 -0.01  0.03 -0.04  0.04  0.01 -0.    0.02  0.03  0.03]


# Exercise 3.1: Static Word Embeddings

**📎 From the slide to the code** · slide *Word Embeddings / Words as vectors*

| In the slide | In the code |
|---|---|
| CBOW: guess a word from its context | `Word2Vec(..., sg=0)` |
| Skip-gram: guess the context from a word | `Word2Vec(..., sg=1)` |
| Pre-trained vectors | `downloader.load('glove-wiki-gigaword-100')`: a smaller model than the slide's |

---

Steps:

Part 1 **Training Word2vec on a dataset**:
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of tokens (a list of tokenized sentences). Use CBOW and a window of 5. Documentation: https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec
- Find the words most similar to, e.g. "state", "states", "president", or others.
- Compare the effectiveness of CBOW and Skipgram. Which one works better?


Part 2 **Using pre-trained static models**:
- Load a pre-trained gensim model (available models: https://github.com/RaRe-Technologies/gensim-data#Models) (Choose a small one to avoid waiting for the download, e.g. `glove-wiki-gigaword-100`).
- Evaluate the following relations:
  - king + woman - man = ?
  - France - Paris + London = ?
  - Find out some for yourselves

**Question: Find the words most similar to "italy" using the 2 models (trained and pretrained). What do you observe? Why? Any solution?**



In [10]:
nltk.download('punkt') ###< run once
nltk.download('punkt_tab') ###< run once, needed by recent NLTK versions

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**Data preparation (given).** Word2Vec learns from the words *around* each word, so it needs **one list of words per sentence**, not per speech. The next cells build it: `corpus` is the input for Word2Vec, and you will use it also in Exercises 3.2 and 3.3.

In [11]:
data['sent_tokens'] = data.text.str.lower()

data.sent_tokens = data.sent_tokens.str.replace(r'\d+', ' ', regex=True)
data.sent_tokens = data.sent_tokens.apply(lambda input_text : [remove_punctuation(t).split() for t in nltk.sent_tokenize(input_text)])

In [12]:
print('Number of sentences:', len(data.sent_tokens.sum()))

Number of sentences: 5360


In [13]:
corpus = data.sent_tokens.sum() # a list of sentences (each one a list of words), instead of a list of speeches
print(corpus[0][:10])

['fellow', 'citizens', 'of', 'the', 'senate', 'and', 'of', 'the', 'house', 'of']


In [14]:
# Your code here

from gensim import downloader
from gensim.models import Word2Vec

> **Note: your results may differ.** `seed=42` fixes the starting point, but gensim trains with several threads in parallel, and the order in which the threads work changes at every run. So the similar words (and their scores) can change a little from one run to another, and from the ones of your classmates. The same holds for `Doc2Vec` in Exercise 3.4.
>
> Look at the *kind* of words you get, not at the exact list.

### How to think about it

**Goal:** train word vectors on the speeches, compare CBOW and skip-gram, and compare them with a pre-trained model.

1. **The data.** Word2Vec learns from the words *around* each word, so it needs a list of sentences, each one a list of words: it is `corpus`, given above.
2. **CBOW vs skip-gram.** Train two models on the same `corpus` that differ only in `sg` (0 = CBOW, 1 = skip-gram). Then ask both for the words most similar to the same word (e.g. `states`): if the lists differ, it is only because of the architecture.
3. **A pre-trained model.** Load GloVe with `downloader.load`: it is used like `model.wv` (`wiki_model[word]`, `wiki_model.most_similar(...)`). An analogy is a sum of vectors: the words in `positive` are added, the ones in `negative` subtracted.
4. **`italy`.** Before asking for its neighbours, check whether the word is in the vocabulary of each model (`'italy' in model.wv`). A model knows only the words it has seen in training: what was in the speeches, what was in Wikipedia?

**Watch out:** the results change a little at every run (see the note above): compare the *kind* of words, not the exact lists.

### Solution 3.1

In [15]:
# sg=0 for CBOW, sg=1 for skip-gram; window=5: 5 words on each side are the context
cbow_model = Word2Vec(sentences=corpus, vector_size=100, window=5, sg=0, seed=42)
model = Word2Vec(sentences=corpus, vector_size=100, window=5, sg=1, seed=42)

# the 8 closest words to 'states' in each model (we keep only the words, not the scores)
print('CBOW:     ', [word for word, score in cbow_model.wv.most_similar('states', topn=8)])
print('Skip-gram:', [word for word, score in model.wv.most_similar('states', topn=8)])

CBOW:      ['of', 'the', 'united', 'constitution', 'government', 'world', 'people', 'and']
Skip-gram: ['constitution', 'powers', 'defend', 'protect', 'authority', 'reserved', 'members', 'federal']


In [16]:
# 'italy' never appears in the speeches, so it has no vector (most_similar('italy') would give a KeyError)
print("Is 'italy' in the vocabulary?", 'italy' in model.wv)

Is 'italy' in the vocabulary? False


In [17]:
# the names of the pre-trained models that gensim can download
downloader.info()['models'].keys()

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

In [18]:
# GloVe trained on Wikipedia + Gigaword (news), 100 numbers per word: about 128 MB to download
wiki_model = downloader.load('glove-wiki-gigaword-100')

In [19]:
# a pre-trained model is used like model.wv: wiki_model[word] is the vector of the word
president_vector = wiki_model['president']
print('Vector size:', president_vector.shape)
print('First 5 values:', president_vector[:5])

Vector size: (100,)
First 5 values: [-0.064549 -0.13812   0.50017   0.41434   0.45832 ]


In [20]:
# king + woman - man: the positive words are added, the negative ones subtracted
wiki_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)

[('queen', 0.7698540687561035),
 ('monarch', 0.6843381524085999),
 ('throne', 0.6755736470222473),
 ('daughter', 0.6594556570053101),
 ('princess', 0.6520534157752991)]

In [21]:
# france - paris + london: "the country whose capital is london"
# (GloVe is lowercase: 'France' would give a KeyError)
wiki_model.most_similar(positive=['france', 'london'], negative=['paris'], topn=5)

[('britain', 0.8921419978141785),
 ('england', 0.8002905249595642),
 ('australia', 0.7940487265586853),
 ('scotland', 0.7838966846466064),
 ('british', 0.7608305215835571)]

In [22]:
# 'italy' is not in our speeches, but it is in the pre-trained model (trained on Wikipedia)
wiki_model.most_similar('italy', topn=5)

[('spain', 0.7746186256408691),
 ('italian', 0.7569283246994019),
 ('portugal', 0.7421525716781616),
 ('germany', 0.740085244178772),
 ('greece', 0.7235245108604431)]

### Answers

**CBOW or skip-gram?** Look at the words most similar to `states`:
- CBOW returns many stop words (`the`, `of`, `in`): they appear next to `states` very often, and CBOW learns mostly from frequent words.
- Skip-gram returns words related to the meaning (`constitution`, `powers`, `federal`, `territories`).

On a small corpus (about 5,000 sentences) **skip-gram works better**, because it learns a good vector also for the rare words.

**What about `italy`?**
- Trained model: `italy` never appears in the speeches, so it has **no vector** (`'italy' in model.wv` is `False`, and `most_similar('italy')` gives a `KeyError`).
- Pre-trained GloVe: it was trained on Wikipedia, so it knows `italy`, and the most similar words are other countries (`spain`, `portugal`, `germany`, `greece`) and `italian`.

**Solutions:** use a pre-trained model, trained on a much larger corpus; or a model that builds the vector from pieces of the word (e.g. *fastText*, `fasttext-wiki-news-subwords-300`), which gives a vector also to words it has never seen.

# Exercise 3.2: Embeddings with multiword ngrams

**📎 From the slide to the code** · slide *Words as vectors*

| In the slide | In the code |
|---|---|
| Frequent pairs become one token (New_York_Times) | `Phrases(corpus)`: `united states` → `united_states` |

---

**Training Word2vec on a dataset**:
- Train a bigram detector using gensim.
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of bigram tokens (a list of tokenized sentences).
- Find the words most similar to, e.g. "united_states", "american_people" or others.
- Try both word2vec architectures.

**Question: the sparsity problem.**
1. How many times do `united_states` and `american_people` appear in the bigram corpus? And `american` and `people` alone?
2. Look at the words most similar to `american_people`, and at their scores. Do they make sense?
3. Why does joining two words into one token make the vectors worse? What could fix it?

*Hint: see `Phrases` in the Introduction to gensim, and apply it to `corpus` (Exercise 3.1). On the real corpus keep the default `min_count` and `threshold`. To count how many times a bigram appears, put all the tokens in a list and use `pd.Series(list).value_counts()`.*

In [23]:
# Your code here

### How to think about it

**Goal:** see what happens to the vectors when frequent pairs of words become a single token.

1. **Find the bigrams.** `Phrases(corpus)` learns which pairs of words appear together much more often than by chance. Applied to the sentences (`bigram_generator[corpus]`), it joins them into one token: `united`, `states` → `united_states`.
2. **Count.** Count how many times each token appears in the new corpus. These counts are the number of **examples** Word2Vec has to learn each vector.
3. **Train.** Same code as in Exercise 3.1, with the bigram sentences as corpus, once with skip-gram and once with CBOW.
4. **Look at the neighbours and at their scores**, for a frequent bigram (`united_states`) and a rarer one (`american_people`). Do the neighbours make sense? Are the scores all similar? Link what you see to the counts of step 2.

**Watch out:** a bigram is always rarer than the two words it is made of: `american_people` takes its occurrences away from `american` and `people`.

### Solution 3.2

In [24]:
from gensim.models import Phrases

# 1. learn which pairs of words appear together more often than by chance (same corpus as in Exercise 3.1)
bigram_generator = Phrases(corpus)

In [25]:
# 1. apply it: in every sentence the detected pairs become one token, e.g. 'united', 'states' -> 'united_states'
bigram_tokens = list(bigram_generator[corpus])

# 2. how many times does each token appear? (fewer occurrences = less to learn from)
bigram_counts = pd.Series([w for sentence in bigram_tokens for w in sentence]).value_counts()
print(bigram_counts[['united_states', 'american_people', 'american', 'people']])

united_states      166
american_people     41
american           124
people             561
Name: count, dtype: int64


In [26]:
# 3. same code as in Exercise 3.1, with the bigram sentences as corpus (skip-gram)
model = Word2Vec(sentences=bigram_tokens, vector_size=100, window=5, sg=1, seed=42)

In [27]:
# 4. united_states appears often enough: its neighbours are related to it
print(model.wv.most_similar('united_states', topn=10))

[('constitution', 0.9693021774291992), ('body', 0.9556083679199219), ('powers', 0.9543904066085815), ('authority', 0.9521089792251587), ('legislative', 0.9491411447525024), ('first', 0.9488481283187866), ('departments', 0.9480668902397156), ('policy', 0.947239875793457), ('protect', 0.9458853006362915), ('whole', 0.9452949166297913)]


In [28]:
# 4. american_people appears much less: look at the neighbours and at their scores
print(model.wv.most_similar('american_people', topn=10))

[('hand', 0.9921948909759521), ('presidential', 0.9906723499298096), ('station', 0.9904929399490356), ('prayers', 0.9904925227165222), ('obligation', 0.9895374178886414), ('election', 0.9881854057312012), ('performance', 0.9878679513931274), ('elected', 0.9878168702125549), ('commitment', 0.98738032579422), ('sacred', 0.98702073097229)]


In [29]:
# 3. The same with CBOW (sg=0)
cbow_bigram_model = Word2Vec(sentences=bigram_tokens, vector_size=100, window=5, sg=0, seed=42)
print('CBOW:     ', [w for w, s in cbow_bigram_model.wv.most_similar('american_people', topn=8)])
print('Skip-gram:', [w for w, s in model.wv.most_similar('american_people', topn=8)])

# 4. How similar are the neighbours? All almost the same score = the model cannot tell them apart
print('Skip-gram scores:', [round(s, 3) for w, s in model.wv.most_similar('american_people', topn=8)])

CBOW:      ['present', 'over', 'office', 'industrial', 'high', 'support', 'through', 'share']
Skip-gram: ['hand', 'presidential', 'station', 'prayers', 'obligation', 'election', 'performance', 'elected']
Skip-gram scores: [0.992, 0.991, 0.99, 0.99, 0.99, 0.988, 0.988, 0.988]


### Answer: the sparsity problem

Word2Vec learns the vector of a word from the contexts where it appears: **few occurrences = few examples = a bad vector**. This is the *sparsity problem*.

**1.** Joining two words into one token splits their occurrences:

| token | before `Phrases` | after `Phrases` |
|---|---|---|
| `american` | 184 | 124 |
| `people` | 602 | 561 |
| `american_people` | — | 41 |
| `united_states` | — | 166 |

`american_people` has only **41** examples to learn from, against 561 for `people`.

**2.** No: the neighbours of `american_people` change at every run and have nothing to do with its meaning, and their scores are all almost the same (≈ 0.99): the model cannot tell them apart. `united_states` (166 times) gets more related neighbours, such as `constitution` and `powers`.

**3.** Every new token is rarer than the words it is made of. The problem exists also without bigrams: of the 9,371 different words of the speeches, **6,593 appear less than 5 times** and Word2Vec (`min_count=5`) gives them no vector at all. To fix it we need **more data**: a bigger corpus, or a model pre-trained on a huge one (e.g. GloVe, Exercise 3.1).

# Exercise 3.3

**📎 From the slide to the code** · slide *Doc2Vec*

| In the slide | In the code |
|---|---|
| First idea: average the word vectors | sum of `model.wv[word]` over the words of a speech |
| Better: weight them with TF-IDF | `model.wv[word] * docs_tfidf[doc_id, term_index[word]]` |

You can represent a document as the weighted sum of the word embeddings of the terms it contains, where the term weights are given by, e.g. Tf-Idf.

- Compute the document embeddings for the US presidential speeches corpus.
- Write a function that, given a document (speech) id, finds the top n most similar documents (as `most_similar` does for words).


The data you need:
- `data.preprocessed`: the text of each speech (a string), to fit the TF-IDF vectorizer;
- `data.tokens`: the words of each speech (a list), to build its embedding.

The next cell is given: it prepares the TF-IDF matrix and a skip-gram model. Then complete `get_document_embedding` (replace the `___`).

Hints (but there may be other / better solutions):

- To obtain the TfIdf values from the 'tfidf' matrix returned by fit_transform, use 'tfidf[doc_id, term_index]'. The list of terms (features) is returned by 'vectorizer.get_feature_names_out()'
- To obtain the Word2Vec embedding for a word, use 'model.wv[word]', where 'model = Word2Vec(...)'
- To obtain the document embedding, iterate over all of its words (that are present both as features for the TfIdf vectorizer and in the Word2Vec model vocabulary), and sum over all the products: (model.wv[word] * tfidf[doc_id, word_index])

In [30]:
# Given: TF-IDF of the speeches and a skip-gram Word2Vec model (trained on the sentences of Exercise 3.1)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
docs_tfidf = vectorizer.fit_transform(data.preprocessed)
model = Word2Vec(sentences=corpus, window=5, sg=1, seed=42)

# position of each word in the TF-IDF matrix: docs_tfidf[doc_id, term_index[word]]
term_index = {word: i for i, word in enumerate(vectorizer.get_feature_names_out())}

In [31]:
# Complete the function: replace the ___
def get_document_embedding(doc_id):
    embedding = np.zeros(model.vector_size)
    for word in set(data.tokens[doc_id]):   # each word once: the TF-IDF weight already counts its repetitions
        if word in term_index and word in model.wv:
            embedding = embedding + ___ * ___
    return embedding

In [32]:
# Your code here
# 1. compute the embedding of every speech (a matrix with one row per speech)
# 2. compute the cosine similarity between all the speeches
# 3. write top_n_most_similar_document(doc_id, doc_sim, top_n=5): the year and president of the most similar speeches
#    (doc_sim = the similarity matrix: so you can reuse the function in Exercise 3.4)

### How to think about it

**Goal:** turn each speech into one vector, then find the speeches most similar to a given one.

1. **One vector per speech.** A speech is a set of words. Each word has a vector (`model.wv[word]`) and a weight in this speech (its TF-IDF). Multiply each vector by its weight and add them up: the rare words, typical of this speech, count more than the common ones. Do it for every speech: a matrix with one row per speech (60 × 100).
2. **Compare all the speeches.** The cosine between two rows says how similar two speeches are. `cosine_similarity` on the whole matrix gives a 60 × 60 table: row `i` = the similarities of speech `i` with all the others.
3. **The most similar speeches.** Take the row of the speech, sort it from the highest value and skip the first one: it is the speech itself (similarity 1).
4. **Check the result.** Before trusting the ranking, look at the similarities between *different* speeches: are the speeches really different from each other?

**Watch out:** use each word once (`set(...)`): its TF-IDF weight already counts how many times it appears.

### Solution 3.3

In [33]:
# 1. the completed skeleton: the blanks are the word vector and its TF-IDF weight in this speech
def get_document_embedding(doc_id):
    embedding = np.zeros(model.vector_size)
    for word in set(data.tokens[doc_id]):   # each word once: the TF-IDF weight already counts its repetitions
        if word in term_index and word in model.wv:   # only words with both a TF-IDF weight and a vector
            embedding = embedding + model.wv[word] * docs_tfidf[doc_id, term_index[word]]
    return embedding

# 1. one row per speech (the weighted sum above, for every speech)
doc_embeddings = np.array([get_document_embedding(i) for i in range(len(data))])
print(doc_embeddings.shape)

# 2. cosine similarity between all the speeches: doc_cos[i, j] = similarity of speech i and speech j
doc_cos = cosine_similarity(doc_embeddings)
print(doc_cos.shape)

(60, 100)
(60, 60)


In [34]:
# 3. the most similar speeches
def top_n_most_similar_document(doc_id, doc_sim, top_n=5):
    current_doc_sim = doc_sim[doc_id]                    # the similarities of this speech with all the others
    best = (-current_doc_sim).argsort()[1:(top_n+1)]    # positions sorted from the most similar; position 0 is the document itself
    return data[['year', 'president']].loc[best].assign(similarity=current_doc_sim[best].round(3))

In [35]:
# which speech is number 2?
data.loc[2, ['year', 'president']]

year          1797
president    Adams
Name: 2, dtype: object

In [36]:
# the 5 speeches most similar to speech 2
top_n_most_similar_document(2, doc_cos, 5)

,year,president,similarity
30,1909,Taft,1.000
25,1889,Harrison,1.000
27,1897,McKinley,1.000
23,1881,Garfield,0.999
13,1841,Harrison,0.999


In [37]:
# 4. Are the speeches really different? Look at the similarities between different speeches
others = doc_cos[~np.eye(len(doc_cos), dtype=bool)]   # all the values except the diagonal
print('Min similarity: ', others.min().round(3))
print('Mean similarity:', others.mean().round(3))

Min similarity:  0.964
Mean similarity: 0.995


All the speeches look almost identical to each other: they all use the same frequent words, so the sums of their vectors point in almost the same direction.

A simple fix: subtract the **mean vector** of all the speeches. What is left is what makes each speech different from the others.

In [38]:
# 4. subtract the mean vector of all the speeches: what is left is what makes each speech different
doc_embeddings_centered = doc_embeddings - doc_embeddings.mean(axis=0)
doc_cos_centered = cosine_similarity(doc_embeddings_centered, doc_embeddings_centered)

others = doc_cos_centered[~np.eye(len(doc_cos_centered), dtype=bool)]
print('Min similarity: ', others.min().round(3))
print('Mean similarity:', others.mean().round(3))

# Compare the most similar speeches to Biden 2021, before and after centering
biden_id = data.index[data.year == 2021][0]
display(top_n_most_similar_document(biden_id, doc_cos))
display(top_n_most_similar_document(biden_id, doc_cos_centered))

Min similarity:  -0.989
Mean similarity: -0.012


,year,president,similarity
50,1989,Bush,0.999
39,1945,Roosevelt,0.999
51,1993,Clinton,0.998
48,1981,Reagan,0.998
59,2025,Trump,0.998


,year,president,similarity
50,1989,Bush,0.979
59,2025,Trump,0.951
48,1981,Reagan,0.949
43,1961,Kennedy,0.930
55,2009,Obama,0.929


# Exercise 3.4

**📎 From the slide to the code** · slide *Doc2Vec / How Doc2Vec learns*

| In the slide | In the code |
|---|---|
| Each document gets its own vector | `doc_model.dv[i]` |
| A new document: learn only its tag | `doc_model.infer_vector(words)` |

Use Doc2Vec (https://radimrehurek.com/gensim/models/doc2vec.html) to compute the embeddings for the documents in the presidential speeches dataset. Write a function that finds the most similar documents for a given document(adapt the function of Exercise 3.3).

*Hint: see `Doc2Vec` in the Introduction to gensim. Use `data.tokens` (the words of each speech) to build one `TaggedDocument` per speech, with the row number as tag; after training, `doc_model.dv[i]` is the vector of speech `i`.*

In [39]:
# Your code here

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

### How to think about it

**Goal:** the same task as Exercise 3.3, but with vectors learned directly for each document.

1. **Prepare the documents.** Doc2Vec wants each speech as a `TaggedDocument`: its list of words (`data.tokens`) + a tag, here its row number. The tag works like an extra word that appears in every window of its speech.
2. **Train and read the vectors.** After training, each speech has its own vector, learned together with the word vectors: `doc_model.dv[i]` is the vector of speech `i`.
3. **Compare.** From here it is the same as Exercise 3.3, steps 2–3: cosine similarity between all the speeches, then the same function `top_n_most_similar_document`.
4. **A new text (a query).** It was not in the training, so it has no tag: `infer_vector` learns its vector from its words, keeping the word vectors fixed. Then compare it with all the speeches.

**Watch out:** `infer_vector` wants a list of words (`query.split()`), not a string.

### Solution 3.4

In [40]:
# 1. one TaggedDocument per speech: its words + its row number as tag
documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(data.tokens)]

In [41]:
# 2. train: one vector per speech (100 numbers by default), learned together with the word vectors
doc_model = Doc2Vec(documents, window=5, seed=42)

In [42]:
# 2. the vector learned for each speech during training, looked up by its tag
d2v_doc_embeddings = np.array([doc_model.dv[i] for i in range(data.shape[0])])

# alternative: infer the vector from the words, as for a new document (slower, and random)
# d2v_doc_embeddings = np.array([doc_model.infer_vector(data.tokens[i]) for i in range(data.shape[0])])

In [43]:
# one row per speech, one column per dimension
d2v_doc_embeddings.shape

(60, 100)

In [44]:
# 3. similarity between all the speeches, as in Exercise 3.3
d2v_doc_cos = cosine_similarity(d2v_doc_embeddings, d2v_doc_embeddings)

In [45]:
# 3. same function as in Exercise 3.3, with the Doc2Vec similarities
top_n_most_similar_document(2, d2v_doc_cos)

,year,president,similarity
35,1929,Hoover,0.969
22,1877,Hayes,0.967
24,1885,Cleveland,0.964
14,1845,Polk,0.964
7,1817,Monroe,0.958


In [46]:
ex_query = 'fellow citizens of the united states of america'
# 4. a query is a new document: infer its vector from its words
# infer_vector wants a list of words, not a string
q = doc_model.infer_vector(ex_query.split())

# cosine_similarity wants 2D arrays: reshape(1, -1) turns the vector into a matrix with 1 row
cosine_similarity(d2v_doc_embeddings, q.reshape(1, -1)).shape

(60, 1)

In [47]:
# 4. the speeches most similar to a free-text query
def get_top_n_from_query(query, doc_embeddings, top_n=5):
    q = doc_model.infer_vector(query.split())
    similarities = cosine_similarity(doc_embeddings, q.reshape(1, -1)).flatten()
    # one row per speech, sorted from the most similar
    ranking = data[['year', 'president']].assign(similarity=similarities.round(3))
    return ranking.sort_values('similarity', ascending=False).head(top_n)

# infer_vector is random: we save the result to use it again below
top_docs = get_top_n_from_query(ex_query, d2v_doc_embeddings)
top_docs

,year,president,similarity
59,2025,Trump,0.913
58,2021,Biden,0.908
50,1989,Bush,0.896
43,1961,Kennedy,0.890
45,1969,Nixon,0.887


In [48]:
# beginning of the most similar speech: does it really look like the query?
top_id = top_docs.index[0]
print(data.loc[top_id, 'year'], data.loc[top_id, 'president'])
print(data.loc[top_id, 'text'][:200], '...')

2025 Trump
Thank you. Thank you very much, everybody. Wow. Thank you very, very much.

Vice President Vance, Speaker Johnson, Senator Thune, Chief Justice Roberts, justices of the Supreme Court of the United Sta ...
